# Inference routing on agentgateway

Route LLM traffic to a self-hosted model pool and let agentgateway pick which replica serves each request from the model servers' live load. It hands every request to an **Endpoint Picker (EPP)** that scores the replicas on three signals: how full each one's KV cache is, how deep its request queue is, and which replica already holds this prompt's prefix in GPU memory (so it can skip prefill). Route to the warm replica and you reuse work; route to the wrong one and you pay for prefill again.

This runs on one kind cluster with no GPU. The model servers are the llm-d inference simulator: they speak the OpenAI API and expose the same Prometheus gauges a real vLLM does (`vllm:gpu_cache_usage_perc`, `vllm:num_requests_waiting`), with values we pin so the routing is deterministic and you can flip it while you watch.

<div align="center">

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 560" width="640" style="max-width:100%;height:auto;font-family:-apple-system,Segoe UI,Roboto,sans-serif">
  <defs>
    <marker id="arr" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto">
      <path d="M0,0 L9,4.5 L0,9 z" fill="#64748b"/>
    </marker>
  </defs>
  <rect x="290" y="20" width="180" height="52" rx="8" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.5"/>
  <text x="380" y="51" text-anchor="middle" font-size="16" font-weight="600" fill="#1e293b">Client</text>
  <line x1="380" y1="72" x2="380" y2="112" stroke="#64748b" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="392" y="97" font-size="12.5" fill="#475569" font-family="ui-monospace,Menlo,monospace">POST /v1/chat/completions</text>
  <rect x="256" y="114" width="248" height="56" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.5"/>
  <text x="380" y="138" text-anchor="middle" font-size="16" font-weight="600" fill="#312e81">Gateway (agentgateway)</text>
  <text x="380" y="158" text-anchor="middle" font-size="11.5" fill="#4338ca">HTTPRoute backendRef &#8594; InferencePool</text>
  <line x1="380" y1="170" x2="380" y2="212" stroke="#64748b" stroke-width="1.5" marker-end="url(#arr)"/>
  <rect x="250" y="214" width="260" height="60" rx="8" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.5"/>
  <text x="380" y="239" text-anchor="middle" font-size="15" font-weight="600" fill="#1e293b">Endpoint Picker (EPP)</text>
  <text x="380" y="259" text-anchor="middle" font-size="12" fill="#475569">scores each replica</text>
  <line x1="510" y1="244" x2="560" y2="244" stroke="#64748b" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="535" y="236" text-anchor="middle" font-size="11" fill="#475569">scrapes</text>
  <rect x="560" y="216" width="188" height="56" rx="8" fill="#f8fafc" stroke="#cbd5e1" stroke-width="1.5"/>
  <text x="654" y="240" text-anchor="middle" font-size="11.5" fill="#334155" font-family="ui-monospace,Menlo,monospace">gpu_cache_usage_perc</text>
  <text x="654" y="258" text-anchor="middle" font-size="11.5" fill="#334155" font-family="ui-monospace,Menlo,monospace">num_requests_waiting</text>
  <line x1="380" y1="274" x2="380" y2="316" stroke="#64748b" stroke-width="1.5"/>
  <text x="392" y="300" font-size="12" fill="#475569">picks the replica with the best score</text>
  <path d="M380,316 L380,332 L215,332 L215,360" fill="none" stroke="#64748b" stroke-width="1.5" marker-end="url(#arr)"/>
  <path d="M380,316 L380,332 L545,332 L545,360" fill="none" stroke="#64748b" stroke-width="1.5" marker-end="url(#arr)"/>
  <rect x="120" y="362" width="190" height="60" rx="8" fill="#e0f2fe" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="215" y="387" text-anchor="middle" font-size="15" font-weight="600" fill="#075985">pool-a</text>
  <text x="215" y="407" text-anchor="middle" font-size="12" fill="#0369a1">cold &#183; cache free, no queue</text>
  <rect x="450" y="362" width="190" height="60" rx="8" fill="#fee2e2" stroke="#f87171" stroke-width="1.5"/>
  <text x="545" y="387" text-anchor="middle" font-size="15" font-weight="600" fill="#991b1b">pool-b</text>
  <text x="545" y="407" text-anchor="middle" font-size="12" fill="#b91c1c">hot &#183; cache full, queue backing up</text>
</svg>

</div>

The pieces: agentgateway with the Gateway API Inference Extension enabled, an `InferencePool` (the routable pool of model servers), the Endpoint Picker that scores replicas, and `InferenceObjective` for serving priority.

## Connect

The inference cluster is already up (`./demo-scripts/setup-all-labs.sh` brings it up with the rest of the suite, or `./scripts/quick.sh up` for this lab on its own). Run the cells with the Bash kernel. This points at the cluster and opens one port-forward to the gateway that every request below reuses.

In [ ]:
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
export CTX=kind-inference NS=inference
kubectl --context $CTX -n $NS get inferencepool vllm-sim >/dev/null 2>&1 \
  && echo "inference lab up on $CTX" || echo "not up: run ./scripts/quick.sh up"
# one port-forward to the gateway, reused by every request in this notebook
pkill -f "port-forward.*inference-gateway" 2>/dev/null; sleep 1
kubectl --context $CTX -n $NS port-forward svc/inference-gateway 18080:80 >/tmp/agw-inf-pf.log 2>&1 &
sleep 3; echo "gateway ready at localhost:18080"

## The model pool

Two model-server replicas, both labelled `app=vllm-sim` so the `InferencePool` selects them. The HTTPRoute's backend is the `InferencePool`, not a Service, so agentgateway hands the endpoint choice to the picker. Each replica pins its cache and queue gauges from a mounted config. Out of the box **pool-a is cold** (KV cache 10% used, empty queue) and **pool-b is hot** (90% used, 8 waiting).

In [ ]:
kubectl --context $CTX -n $NS get pods -l app=vllm-sim -o wide
echo; echo "the route sends the pool's traffic to the InferencePool, not a Service:"
kubectl --context $CTX -n $NS get httproute llm-route -o jsonpath='{.spec.rules[0].backendRefs[0]}'; echo
kubectl --context $CTX -n $NS get inferencepool vllm-sim
echo; echo "pinned signals:"
echo "  pool-a: $(kubectl --context $CTX -n $NS get cm sim-pool-a -o jsonpath='{.data.config\.yaml}' | tr '\n' ' ')"
echo "  pool-b: $(kubectl --context $CTX -n $NS get cm sim-pool-b -o jsonpath='{.data.config\.yaml}' | tr '\n' ' ')"

## The Endpoint Picker (EPP)

The EPP is the brain. The `InferencePool` points at it with `endpointPickerRef`, and agentgateway calls it (over gRPC) for **every** request to choose a replica. It scores each replica with a weighted set of plugins. Read its config straight off the cluster, this is exactly how it decides:

In [ ]:
kubectl --context $CTX -n $NS get cm vllm-sim-epp -o jsonpath='{.data.default-plugins\.yaml}'
echo
echo "so each replica is scored on: prefix-cache affinity (weight 3), KV-cache use (2), queue depth (2)."
echo "InferencePool -> EPP wiring:"
kubectl --context $CTX -n $NS get inferencepool vllm-sim -o jsonpath='  endpointPickerRef: {.spec.endpointPickerRef.name}:{.spec.endpointPickerRef.port.number}'; echo

### Reading the weights

The picker gives every replica a score and routes to the highest. Each scorer votes, and its **weight** is how much its vote counts: here prefix-cache ×3, kv-cache ×2, queue ×2.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 780 430" width="780" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,0 L9,4.5 L0,9 z" fill="#64748b"/></marker><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,0 L9,4.5 L0,9 z" fill="#16a34a"/></marker></defs><rect x="0" y="0" width="780" height="430" rx="10" fill="#f8fafc"/><text x="390" y="30" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">The picker scores every replica, then routes to the highest weighted total</text><rect x="24" y="196" width="112" height="56" rx="8" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.5"/><text x="80" y="222" text-anchor="middle" font-size="13" font-weight="700" fill="#1e293b">request</text><text x="80" y="240" text-anchor="middle" font-size="9.5" fill="#64748b">one prompt</text><line x1="136" y1="224" x2="176" y2="224" stroke="#64748b" stroke-width="1.8" marker-end="url(#a)"/><rect x="178" y="74" width="318" height="300" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.6"/><text x="337" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#312e81">Endpoint Picker · weighted vote</text><rect x="196" y="116" width="282" height="70" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="212" y="140" font-size="12.5" font-weight="700" fill="#14532d">prefix-cache</text><text x="212" y="159" font-size="9.5" fill="#166534">already holds this prompt's prefix?</text><text x="212" y="174" font-size="9" fill="#166534" font-style="italic">skip prefill = biggest win</text><circle cx="450" cy="151" r="19" fill="#16a34a"/><text x="450" y="157" text-anchor="middle" font-size="15" font-weight="700" fill="#fff">×3</text><rect x="196" y="196" width="282" height="62" rx="7" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.4"/><text x="212" y="219" font-size="12.5" font-weight="700" fill="#1e293b">kv-cache use</text><text x="212" y="237" font-size="9.5" fill="#475569">how full is its GPU cache?</text><circle cx="450" cy="227" r="17" fill="#64748b"/><text x="450" y="233" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">×2</text><rect x="196" y="268" width="282" height="62" rx="7" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.4"/><text x="212" y="291" font-size="12.5" font-weight="700" fill="#1e293b">queue depth</text><text x="212" y="309" font-size="9.5" fill="#475569">how many requests are waiting?</text><circle cx="450" cy="299" r="17" fill="#64748b"/><text x="450" y="305" text-anchor="middle" font-size="14" font-weight="700" fill="#fff">×2</text><text x="337" y="356" text-anchor="middle" font-size="10" fill="#4338ca" font-style="italic">weighted sum per replica · highest wins</text><line x1="496" y1="150" x2="554" y2="128" stroke="#16a34a" stroke-width="2" marker-end="url(#g)"/><line x1="496" y1="210" x2="554" y2="205" stroke="#cbd5e1" stroke-width="1.6" marker-end="url(#a)"/><line x1="496" y1="250" x2="554" y2="272" stroke="#cbd5e1" stroke-width="1.6" marker-end="url(#a)"/><rect x="556" y="102" width="200" height="62" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="656" y="125" text-anchor="middle" font-size="13" font-weight="700" fill="#14532d">replica A · 4.0 ✓</text><text x="656" y="145" text-anchor="middle" font-size="9.5" fill="#166534">has the prefix (wins on ×3)</text><rect x="556" y="178" width="200" height="52" rx="8" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.4"/><text x="656" y="200" text-anchor="middle" font-size="12.5" font-weight="700" fill="#334155">replica B · 3.4</text><text x="656" y="217" text-anchor="middle" font-size="9.5" fill="#64748b">free cache, empty queue</text><rect x="556" y="244" width="200" height="52" rx="8" fill="#eef2f7" stroke="#94a3b8" stroke-width="1.4"/><text x="656" y="266" text-anchor="middle" font-size="12.5" font-weight="700" fill="#334155">replica C · 2.4</text><text x="656" y="283" text-anchor="middle" font-size="9.5" fill="#64748b">middling on both</text><text x="390" y="404" text-anchor="middle" font-size="10.5" fill="#475569" font-style="italic">prefix-cache is weighted ×3: a cached prefix skips prefill, the biggest latency win. With no prefix hit, kv-cache and queue (×2) spread the load.</text></svg></div>

- **prefix-cache (×3)**: does this replica already hold the prompt's prefix? Reusing it skips prefill, the biggest latency win, so it counts most.
- **kv-cache use (×2)**: how full is the replica's cache? Steer away from the ones near capacity.
- **queue depth (×2)**: how many requests are already waiting? Avoid the backed-up ones.

Worked example, each judge scores 0 (bad) to 1 (good), then × its weight:

| replica | prefix ×3 | kv-cache ×2 | queue ×2 | total |
|---|---|---|---|---|
| **A**: has the prefix, 90% full, 8 queued | 1.0 → 3.0 | 0.2 → 0.4 | 0.3 → 0.6 | **4.0 ✓** |
| **B**: no prefix, 30% full, empty queue | 0.0 | 0.7 → 1.4 | 1.0 → 2.0 | 3.4 |
| **C**: no prefix, 50% full, 2 queued | 0.0 | 0.5 → 1.0 | 0.7 → 1.4 | 2.4 |

A wins: a cached prefix (×3) beats a shorter queue. With no prefix hit, kv-cache and queue decide.

## Two commands we reuse

`signals` prints what the picker sees on each replica (its pinned KV-cache and queue). `route_test N` fires N chat-completions at the gateway and tallies which replica actually served them, read straight from the gateway's own access log (the `selected_endpoint` it routed each request to). `set_metrics` re-pins a replica's gauges so we can move the decision on cue. They are plain kubectl and curl, nothing hidden:

In [ ]:
signals() {
  for r in a b; do
    cfg=$(kubectl --context $CTX -n $NS get cm sim-pool-$r -o jsonpath='{.data.config\.yaml}')
    printf '  pool-%s: kv-cache=%s  queue=%s\n' "$r" \
      "$(echo "$cfg" | awk '/kv-cache-usage/{print $2}')" \
      "$(echo "$cfg" | awk '/waiting-requests/{print $2}')"
  done
}
# which endpoint the gateway routed to, straight from its access log (selected_endpoint)
_gw() { kubectl --context $CTX -n $NS logs deploy/inference-gateway --tail=-1 2>/dev/null | grep -c "selected_endpoint=$1:"; }
route_test() {
  local n="${1:-8}" i
  local aip=$(kubectl --context $CTX -n $NS get pod -l replica=pool-a -o jsonpath='{.items[0].status.podIP}')
  local bip=$(kubectl --context $CTX -n $NS get pod -l replica=pool-b -o jsonpath='{.items[0].status.podIP}')
  local a0=$(_gw "$aip") b0=$(_gw "$bip")
  for i in $(seq 1 "$n"); do
    curl -s -o /dev/null localhost:18080/v1/chat/completions -H 'content-type: application/json' \
      -d '{"model":"base-model","messages":[{"role":"user","content":"Explain Kubernetes in one sentence."}]}'
  done
  sleep 1
  printf '  %s requests -> pool-a served %s, pool-b served %s\n' "$n" "$(( $(_gw "$aip") - a0 ))" "$(( $(_gw "$bip") - b0 ))"
}
set_metrics() {  # <a|b> <kv 0..1> <waiting> <running>
  printf 'model: base-model\nport: 8000\nfake-metrics:\n  kv-cache-usage: %s\n  waiting-requests: %s\n  running-requests: %s\n' "$2" "$3" "$4" \
    | kubectl --context $CTX -n $NS create cm sim-pool-$1 --from-literal=config.yaml="$(cat)" --dry-run=client -o yaml \
    | kubectl --context $CTX -n $NS apply -f - >/dev/null
  kubectl --context $CTX -n $NS rollout restart deploy/vllm-pool-$1 >/dev/null
  kubectl --context $CTX -n $NS rollout status deploy/vllm-pool-$1 --timeout=90s >/dev/null
  printf '  pool-%s -> kv-cache=%s queue=%s\n' "$1" "$2" "$3"
}
echo "helpers ready: signals | route_test N | set_metrics <a|b> <kv> <waiting> <running>"

## 1. KV-cache-aware routing

pool-a is cold, pool-b is hot. Fire eight requests: with pool-b's cache nearly full, the picker sends everything to pool-a, the replica that can serve without evicting cache.

In [ ]:
signals
route_test 8

### Flip it on cue

Saturate pool-a and free pool-b. Nothing about the route changes. The picker re-scores on the new gauges and traffic follows to pool-b.

In [ ]:
set_metrics a 0.95 9 6    # pool-a now HOT
set_metrics b 0.05 0 1    # pool-b now COLD
sleep 6                   # let the picker re-scrape
signals
route_test 8              # traffic has moved to pool-b

## 2. Queue-aware routing

KV cache is not the only signal. Give both replicas the same cache but back pool-a's queue up (30 waiting). The queue-scorer penalises it and traffic goes to pool-b, the one that can start work now. This is what keeps tail latency down when a replica is head-of-line blocked.

In [ ]:
set_metrics a 0.50 30 6   # cache free, but 30 requests queued
set_metrics b 0.50 0 1    # cache free, empty queue
sleep 6
signals
route_test 8              # avoids pool-a's queue

## 3. Prefix-cache affinity

The highest-weighted scorer (weight 3 in the EPP config above). When many requests share a prompt prefix, a large system prompt, a shared document, a long few-shot preamble, the picker keeps them on the replica that already holds that prefix in cache, so it skips prefill entirely. It outweighs KV-cache and queue because reusing a cached prefix is the single biggest latency win in serving. In production this is where most of the speed-up comes from; the KV and queue scorers then break ties and steer around saturation.

## 4. Serving priority with InferenceObjective

`InferenceObjective` attaches a priority to requests against the pool. Under contention the scheduler drains higher-priority work first, so interactive traffic keeps flowing while batch work waits. Two tiers here: `interactive` (priority 10) and `batch` (priority 0). Priority bites when the pool is saturated, not when it is idle.

In [ ]:
kubectl --context $CTX -n $NS get inferenceobjective

## Reset

Put both replicas back to their starting state (pool-a cold, pool-b hot) for the next run.

In [ ]:
set_metrics a 0.10 0 1
set_metrics b 0.90 8 5
echo "reset: pool-a cold, pool-b hot"

Tear the whole thing down with `./scripts/quick.sh teardown`.